# IGANN: Interpretable Generalized Additive Neural Network

IGANN begins with a sparse linear model and stagewise fits feature-wise extreme-learning-machine corrections. Random hidden weights stay fixed; only output coefficients are solved.


## Model


$$
F_T(x)=\beta_0+\sum_j\beta_jx_j+
\eta\sum_{t=1}^{T}\sum_j w_{tj}^{\mathsf T}\sigma(a_{tj}x_j).
$$

The stagewise residual fit lets a feature remain linear unless nonlinear corrections improve the objective.


## Shared estimator API

All neural estimators use `fit`, `predict`, `score`, `evaluate`, and
`predict_components`. The component result reconstructs predictions on the link
scale and supports shared term-importance and plotting utilities. Constructor
options such as `numerical_preprocessing` and `categorical_preprocessing` are forwarded to
PreTab and are fitted on training rows only.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(7)
n = 180
X = pd.DataFrame({
    "x1": rng.uniform(-1.0, 1.0, n),
    "x2": rng.normal(size=n),
    "group": rng.choice(["a", "b", "c"], size=n),
})
y = (
    np.sin(np.pi * X["x1"])
    + 0.35 * X["x2"] ** 2
    + 0.30 * (X["group"] == "b")
    + rng.normal(0.0, 0.12, n)
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7
)

# Set True to run the small fit and all fitted-model demonstrations.
RUN_TRAINING = False


## Construct the estimator


In [ ]:
from nampy.models import IGANNClassifier, IGANNLSS, IGANNRegressor


model = IGANNRegressor(
    n_hid=10,
    n_estimators=80,
    boost_rate=0.1,
    early_stopping=12,
    solver="native",
    sparse=0,
)
model.get_params(deep=False)


## Fit and inspect

Enable `RUN_TRAINING` above for a short demonstration. Real work should use a
larger validation set, enough epochs, and early stopping.


In [ ]:
if RUN_TRAINING:
    model.fit(
        X_train,
        y_train,
        max_epochs=3,
        batch_size=64,
        random_state=7,
        logger=False,
        enable_progress_bar=False,
        enable_model_summary=False,
    )
    predictions = model.predict(X_test)
    r2 = model.score(X_test, y_test)
    metrics = model.evaluate(X_test, y_test)
    components = model.predict_components(X_test, center=True)
    components.validate_additive_reconstruction()
    display({"R2": r2, **metrics})
    display(model.term_importance(X_test).head())


## Model-specific controls

`solver='native'` requests the upstream-style stagewise optimizer. `sparse>0` enables ABESS feature selection and requires the optional `igann-sparse` dependency.


In [ ]:
if RUN_TRAINING:
    display(model.training_history())
    display(model.selected_features_)
    display(model.basis_metadata())
    display(model.model_complexity())


## Task variants and limits

Native training supports regression and binary classification. Multiclass `IGANNClassifier` and `IGANNLSS` use the fixed basis with the shared gradient engine.
